In [ ]:
import pandas as pd

df = pd.read_csv('furniture_10k_FINAL.csv', sep=';')

print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   sales_date        10000 non-null  object
 1   order_id          10000 non-null  object
 2   customer_name     10000 non-null  object
 3   product_name      10000 non-null  object
 4   category          10000 non-null  object
 5   price             10000 non-null  int64 
 6   quantity          10000 non-null  int64 
 7   discount          10000 non-null  object
 8   total             10000 non-null  int64 
 9   shipping_fee      10000 non-null  int64 
 10  total_sales       10000 non-null  int64 
 11  status            10000 non-null  object
 12  shipping_address  10000 non-null  object
dtypes: int64(5), object(8)
memory usage: 1015.8+ KB
None


In [ ]:
# Jumlah data sebelum cleaning
print("Jumlah data sebelum cleaning:", len(df))

# Cek missing values
print("\nMissing Values:")
print(df.isnull().sum())

# Cek duplicate rows
duplicate_count = df.duplicated().sum()
print("\nJumlah duplicate rows:", duplicate_count)

# Hapus duplicate rows
df = df.drop_duplicates()

# Hapus missing values (jika ada)
df = df.dropna()

# Jumlah data sesudah cleaning
print("\nJumlah data sesudah cleaning:", len(df))

Jumlah data sebelum cleaning: 10000

Missing Values:
sales_date          0
order_id            0
customer_name       0
product_name        0
category            0
price               0
quantity            0
discount            0
total               0
shipping_fee        0
total_sales         0
status              0
shipping_address    0
dtype: int64

Jumlah duplicate rows: 0

Jumlah data sesudah cleaning: 10000


In [ ]:
# Membuat kategori revenue tier
df['revenue_tier'] = pd.cut(
    df['total_sales'],
    bins=[0, 500000, 2000000, float('inf')],
    labels=['Low', 'Mid', 'High']
)

# Menampilkan sample hasil
print(df[['total_sales', 'revenue_tier']].head())

# Menampilkan jumlah tiap kategori
print(df['revenue_tier'].value_counts())

   total_sales revenue_tier
0      9560000         High
1      2060000         High
2      2650000         High
3     16550000         High
4      9660000         High
revenue_tier
High    7645
Mid     2355
Low        0
Name: count, dtype: int64


In [ ]:
# Membuat kolom kota dari shipping_address
df['kota'] = df['shipping_address'].str.split(', ').str[1]

# Groupby category dan kota
summary = df.groupby(['category', 'kota']).agg({
    'total_sales': 'mean',
    'quantity': 'sum'
}).reset_index()

# Rename kolom agar lebih rapi
summary.rename(columns={
    'total_sales': 'mean_total_sales',
    'quantity': 'total_quantity'
}, inplace=True)

# Tampilkan hasil
print(summary.head(10))

  category               kota  mean_total_sales  total_quantity
0    Dapur             Bekasi      1.046391e+07             165
1    Dapur              Bogor      1.068305e+07             178
2    Dapur              Depok      9.769444e+06             236
3    Dapur      Jakarta Barat      1.174734e+07             189
4    Dapur      Jakarta Pusat      1.058330e+07             180
5    Dapur    Jakarta Selatan      1.118000e+07             188
6    Dapur      Jakarta Timur      1.096802e+07             201
7    Dapur      Jakarta Utara      1.230638e+07             231
8    Dapur          Tangerang      1.031302e+07             174
9    Dapur  Tangerang Selatan      8.421337e+06             188


In [ ]:
print(
    summary.sort_values(
        by='mean_total_sales',
        ascending=False
    ).head(1)
)

  category           kota  mean_total_sales  total_quantity
7    Dapur  Jakarta Utara      1.230638e+07             231


In [ ]:
kota_profit = df.groupby('kota')['total_sales'].mean().sort_values(ascending=False)

print(kota_profit.head(1))

kota
Jakarta Utara    5.663006e+06
Name: total_sales, dtype: float64
